In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import warnings
warnings.filterwarnings('ignore')

print("✅ Bibliotecas carregadas!")

✅ Bibliotecas carregadas!


In [2]:
# Carregar os dados do Cepea
cepea = pd.read_excel('../data/Boi_gordo_corrigido.xlsx', engine='openpyxl')
cepea['Data'] = pd.to_datetime(cepea['Data'], dayfirst=True, errors='coerce')

cepea['Preco_Arroba'] = pd.to_numeric(
    cepea['Preco_Arroba'].astype(str)
    .str.replace(r'R\$', '', regex=True)
    .str.replace(',', '.', regex=False)
    .str.replace('-', '', regex=False)
    .str.strip(), errors='coerce'
)

cepea = cepea.dropna(subset=['Preco_Arroba', 'Data']).sort_values('Data').reset_index(drop=True)

print(f"✅ Dados carregados: {len(cepea)} linhas")
print(f"Período: {cepea['Data'].min().date()} até {cepea['Data'].max().date()}")
cepea.tail()

✅ Dados carregados: 2525 linhas
Período: 2016-01-04 até 2026-03-23


,Data,Preco_Arroba
2520,2026-03-17,351.91
2521,2026-03-18,353.10
2522,2026-03-19,353.88
2523,2026-03-20,354.85
2524,2026-03-23,354.91


In [3]:
# Preparar série temporal
serie = cepea.set_index('Data')['Preco_Arroba']

# Treinar modelo Exponential Smoothing (Holt-Winters)
model = ExponentialSmoothing(
    serie,
    trend='add',
    seasonal='add',
    seasonal_periods=365,   # anual
    use_boxcox=False
)

model_fit = model.fit(optimized=True)

print("✅ Modelo Exponential Smoothing treinado!")

✅ Modelo Exponential Smoothing treinado!


In [4]:
# Previsão para os próximos 90 dias
previsao = model_fit.forecast(90)

# Criar DataFrame da previsão
datas_futuras = pd.date_range(
    start=cepea['Data'].iloc[-1] + pd.Timedelta(days=1),
    periods=90
)

df_previsao = pd.DataFrame({
    'Data': datas_futuras,
    'Preco_Arroba': previsao.values
})

print(f"✅ Previsão gerada para {len(df_previsao)} dias")
df_previsao.head()

✅ Previsão gerada para 90 dias


,Data,Preco_Arroba
0,2026-03-24,355.772157
1,2026-03-25,356.457351
2,2026-03-26,360.473849
3,2026-03-27,359.935032
4,2026-03-28,360.303537


In [5]:
# Gráfico comparativo (últimos 12 meses + previsão)
hoje = cepea['Data'].iloc[-1]
ultimos_12_meses = cepea[cepea['Data'] >= hoje - pd.Timedelta(days=365)]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=ultimos_12_meses['Data'],
    y=ultimos_12_meses['Preco_Arroba'],
    mode='lines',
    name='Histórico (últimos 12 meses)',
    line=dict(color='#1f77b4', width=3)
))

fig.add_trace(go.Scatter(
    x=df_previsao['Data'],
    y=df_previsao['Preco_Arroba'],
    mode='lines',
    name='Previsão 90 dias',
    line=dict(color='#2ca02c', width=4, dash='dash')
))

fig.add_vline(x=hoje, line_dash="dash", line_color="red")
fig.add_annotation(x=hoje, y=1.06, text="HOJE", showarrow=False, font=dict(color="red", size=14))

fig.update_layout(
    title="Previsão dos próximos 90 dias - Exponential Smoothing",
    xaxis_title="Data",
    yaxis_title="Preço por Arroba (R$)",
    hovermode="x unified",
    template="plotly_white",
    height=650,
    legend=dict(orientation="h", y=1.02)
)

fig.show()

In [6]:
# Previsão para +30 dias
preco_30d = df_previsao.iloc[29]['Preco_Arroba']
print(f"🔮 Previsão aproximada para +30 dias: R$ {preco_30d:.2f} por arroba")

🔮 Previsão aproximada para +30 dias: R$ 377.48 por arroba
